# 歷史版本 E：課程提交版（精簡 59 cells）

**用途**：最終版的精簡整理版，章節與 final_analysis/ 對應；保留為一份完整的單檔對照。

| 項目 | 說明 |
|---|---|
| 輸入 | data/raw CSV |
| 輸出 | （歷史探索） |
| 來源 | 原 03_course_code.ipynb 全部 cells |

> 歷史版本，保留供追溯；不保證可獨立執行。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

# 多變量分析期末報告 114-2
* 114354005 林子立
* 114354034 張暠文

## 主題
台灣地區肺癌發生率之長期追蹤分析

## Load data

$$
標準化死亡率= {\sum(標準人口各年齡層之加權數×觀察族群各年齡層死亡率)}
$$

<br>
$\text{標準人口} = $ WHO 2000 年世界標準人口

## Chinese Fonts Setting (for plotting)


## Data Preprocessing

In [ ]:
# rename columns
df.columns = [
    'Diagnosis_Year', 'Gender', 'City_County', 'Cancer_Type',
    'Age_Standardized_Incidence_Rate', 'Cancer_Incidence_Count',
    'Mean_Age', 'Median_Age', 'Crude_Incidence_Rate'
]
# Display the new column names and a preview of the data
print("New column names:", df.columns.tolist())
display(df.head())

In [ ]:
df_filtered = df[(df['Gender'] == '全') &(df['City_County'] != '全國') & (df['Cancer_Type'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='City_County',
    columns='Diagnosis_Year',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='Diagnosis_Year',
    columns='City_County',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
corr_matrix_yr = pivot_table_age_std.corr()
corr_matrix_county = pivot_table_age_std.T.corr()

## Correlation Visualization

* Visualize the reordered correlation matrix (counties) using a heatmap.
* 這裡只放「縣市間」的相關。

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_county, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_county = corr_matrix_county.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_county.head())

In [ ]:
plt.figure(figsize=(12, 10))

ax = sns.heatmap(
    reordered_corr_matrix_county,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=.5,
    annot_kws={'fontsize': 10}
)

plt.title('各縣市年齡標準化發生率相關係數矩陣（依 Hclust 排序）')

plt.savefig('correlation_heatmap_counties.png', dpi=300, bbox_inches='tight')

plt.show()

## EDA (肺癌發生率)

* 折線圖
* 箱型圖

In [ ]:
# 計算字型大小 (依你的需求放大 10%)
base_size = 12 * 1.1
title_size = 16 * 1.1
label_size = 14 * 1.1

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Age_Standardized_Incidence_Rate', hue='City_County')

# 內建全自動套用中文字型，直接帶入大小即可！
plt.title('肺癌 — 年齡標準化發生率 — 按縣市與年度', fontsize=title_size)
plt.xlabel('年', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)

# 圖例也一樣，完全不需 setp 跑迴圈
plt.legend(title='縣市別', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=base_size, title_fontsize=label_size)

plt.grid(True)
plt.tight_layout()
plt.savefig('lung_cancer_lineplot_counties.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(16, 8))
sns.boxplot(data=df_filtered, x='City_County', y='Age_Standardized_Incidence_Rate')
plt.title('肺癌 — 年齡標準化發生率 — 按縣市別 (箱型圖)', fontsize=title_size)
plt.xlabel('縣市別', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)
plt.xticks(rotation=45, ha='right', fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('lung_cancer_boxplot_counties.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(16, 8))

# ✨ 關鍵步驟：計算各縣市的中位數，由小到大排序，並取出縣市名稱 (Index)
county_order = (
    df_filtered.groupby('City_County')['Age_Standardized_Incidence_Rate']
    .median()
    .sort_values()
    .index
)

# ✨ 在 boxplot 中加入 order=county_order
sns.boxplot(
    data=df_filtered,
    x='City_County',
    y='Age_Standardized_Incidence_Rate',
    order=county_order,
    hue='City_County',
    hue_order=county_order,
    palette='coolwarm',
    legend=False
)
sns.despine()

plt.title('肺癌 — 年齡標準化發生率 — 按縣市別 (箱型圖，依中位數排序)', fontsize=title_size)
plt.xlabel('縣市別', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)

# 因為依序排列了，標籤拉直或 45 度都會看起來非常整齊
plt.xticks(rotation=45, ha='right', fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('lung_cancer_boxplot_counties_sorted.png', dpi=300, bbox_inches='tight')
plt.show()

## Data Cleaning

* 排除連江縣

In [ ]:
# 檢查 NaN values 狀況
# Check for rows (counties) with NaN values
rows_with_nan = pivot_table_age_std[pivot_table_age_std.isna().any(axis=1)].index.tolist()

# Check for columns (years) with NaN values
cols_with_nan = pivot_table_age_std.columns[pivot_table_age_std.isna().any(axis=0)].tolist()

print(f"Rows (Counties) with NaN values: {rows_with_nan}")
print(f"Columns (Years) with NaN values: {cols_with_nan}")

# Display the exact locations (Row, Column) where values are missing
if rows_with_nan or cols_with_nan:
    print("\nDetailed NaN locations:")
    nan_locations = pivot_table_age_std.isna().stack()
    nan_locations = nan_locations[nan_locations].index.tolist()
    for row, col in nan_locations:
        print(f"- County: {row}, Year: {col}")
else:
    print("No NaN values found in the table.")

連江縣在 1979 ~ 2000 年間有 11 次出現 NaN

In [ ]:
pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

#### Scree Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 取得特徵值 (λ)
eigenvalues = pca_corr.explained_variance_

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(eigenvalues) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製陡坡圖的折線與點
plt.plot(
    pc_num,
    eigenvalues,
    marker='o',
    linestyle='-',
    color='#2b5c8f',
    linewidth=2,
    markersize=6,
    label='Eigenvalue'
)

# 加上傳統轉折點（Elbow）的輔助標記或 Kaiser 準則線
# 註：如果使用的是「相關係數矩陣(Scaled)」，通常會畫一條 λ = 1 的水平線作為篩選基準
plt.axhline(y=1, color='r', linestyle='--', alpha=0.7, label='Kaiser Criterion (λ = 1)')

plt.title('PCA Scree Plot - Correlation Matrix', fontsize=16)
plt.xlabel('Principal Component', fontsize=13)
plt.ylabel('Eigenvalue (λ)', fontsize=13)

# 讓 X 軸的刻度整齊顯示每一個 PC 編號
plt.xticks(pc_num)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Variance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算累積解釋變異量比率 (乘以 100 轉為百分比)
cum_variance_ratio = np.cumsum(pca_corr.explained_variance_ratio_) * 100

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_variance_ratio) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製累積變異量的折線與點
plt.plot(
    pc_num,
    cum_variance_ratio,
    marker='s',          # 使用方形標記區隔之前的陡坡圖
    linestyle='-',
    color='#e06666',     # 使用顯眼的暖色系
    linewidth=2,
    markersize=6,
    label='Cumulative Explained Variance'
)

# ✨ 加上 80% 的基準輔助線 (通常實務上會看 80% 或 90% 的斷點)
plt.axhline(y=90, color='gray', linestyle='--', alpha=0.7, label='90% Threshold')

# 標註前幾個 PC 的具體數值（讓圖表更直覺）
# 這邊標註前 3 個成分作為範例
for i in range(3):
    plt.annotate(
        f"{cum_variance_ratio[i]:.1f}%",
        (pc_num[i], cum_variance_ratio[i]),
        textcoords="offset points",
        xytext=(0,10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# 圖表細節設定
plt.title('PCA (Correlation) Cumulative Explained Variance', fontsize=16)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

# 讓 X、Y 軸的刻度更貼切
plt.xticks(pc_num)
plt.ylim(0, 105) # Y 軸範圍設在 0~105% 比較有留白美感

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_cumulative_variance_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Biplot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 準備必要數據
# --------------------------------------------------
# (A) 取得 PC Score (縣市投影點在 PC1 和 PC2 的坐標)
scores = pca_corr.transform(scaled_data)
pc1_scores = scores[:, 0]
pc2_scores = -scores[:, 1]
county_names = pivot_table_cleaned.index # 縣市名稱

# (B) 取得 PC Loadings (原始欄位在 PC1 和 PC2 的載荷/箭頭向量)
loadings = pca_corr.components_.T # 轉置成 (num_features, num_components)
pc1_loadings = loadings[:, 0]
pc2_loadings = -loadings[:, 1]
feature_names = pivot_table_cleaned.columns # 原始欄位名稱 (例如：年份)

# (C) 解釋變異量百分比 (對齊前面 R 的 summary 結果)
var1_pct = pca_corr.explained_variance_ratio_[0] * 100
var2_pct = pca_corr.explained_variance_ratio_[1] * 100

# ✨ (D) 計算 Loadings 箭頭的縮放因子 (Scale Factor)
# Biplot 核心技巧：讓 Loading 箭頭長度能夠根據 Score 點的級距動態調整。
# 這裡將 Loadings 最大絕對值拉伸到和 Scores 最大絕對值同一個數量級，再乘上 0.9 (填滿 90% 畫面)
max_score = np.max([np.abs(pc1_scores).max(), np.abs(pc2_scores).max()])
max_loading = np.max([np.abs(pc1_loadings).max(), np.abs(pc2_loadings).max()])
scale_factor = (max_score / max_loading) * 0.9

# 2. 開始繪圖
# --------------------------------------------------
# 確保全域中文字型已設定 (若沒有 prop 變數，請重新執行 Notebook 最上方的字型註冊區塊)
if 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

fig, ax = plt.subplots(figsize=(14*0.8, 11*0.8))

# 📈 (A) 繪製 Score Plot (縣市投影點)
ax.scatter(
    pc1_scores, pc2_scores,
    c='#2b5c8f', alpha=0.6, s=70, edgecolors='white',
    label='縣市投影點'
)

# ✍️ 標註縣市名稱 (中文字)
# 設定小的 xytext 偏移量，避免文字貼在點上
for i, txt in enumerate(county_names):
    ax.annotate(
        txt, (pc1_scores[i], pc2_scores[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#1f3c5f', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的年份箭頭向量)
arrow_color = '#e06666' # 暖色調，與藍點形成對比
for i in range(len(feature_names)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings[i] * scale_factor
    end_y = pc2_loadings[i] * scale_factor

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color, alpha=0.5, linewidth=1,
        head_width=max_score*0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (年份) - 根據箭頭方向微調文字對齊方式，避免疊加
    # 文字放在箭頭終點附近 (加 2% 偏移量)
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(feature_names[i]),
        color=arrow_color, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
# 畫出 PC1、PC2 的原點輔助線
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

# 設定標題和座標軸標籤 (包含解釋變異量比例)
ax.set_title(
    f'PCA Biplot：肺癌年齡標準化發生率 (Correlation)',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1 ', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

# 確保座標軸比例 1:1，避免形變 (這在 PCA 報告中非常重要)
# 若Scores和Loadings級距差異太大，設定 'equal' 有時會導致圖表縮得太小，可依喜好註解掉
# ax.set_aspect('equal')

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_biplot_counties_by_years.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

# 1. 準備必要數據：將 pivot_table_cleaned 轉置，讓年份成為觀測值(rows)，縣市成為變數(columns)
# --------------------------------------------------
data_for_pca_swapped = pivot_table_cleaned.T

# 2. 標準化轉置後的數據 (與 pca_corr 的邏輯一致，使用 StandardScaler)
# --------------------------------------------------
scaler_swapped = StandardScaler()
scaled_data_swapped = scaler_swapped.fit_transform(data_for_pca_swapped)

# 3. 執行 PCA
# --------------------------------------------------
pca_swapped = PCA(n_components=2)
pca_swapped.fit(scaled_data_swapped)

# (A) 取得 PC Score (年份投影點在 PC1 和 PC2 的坐標)
scores_swapped = pca_swapped.transform(scaled_data_swapped)
pc1_scores_swapped = scores_swapped[:, 0]
pc2_scores_swapped = -scores_swapped[:, 1] # 保持與之前 PC2 軸一致的負號翻轉
year_names = data_for_pca_swapped.index # 年份作為點的標籤

# (B) 取得 PC Loadings (縣市在 PC1 和 PC2 的載荷/箭頭向量)
loadings_swapped = pca_swapped.components_.T # 轉置成 (num_features, num_components)
pc1_loadings_swapped = loadings_swapped[:, 0]
pc2_loadings_swapped = -loadings_swapped[:, 1] # 保持與之前 PC2 軸一致的負號翻轉
county_names_swapped = data_for_pca_swapped.columns # 縣市作為箭頭的標籤

# (C) 解釋變異量百分比
var1_pct_swapped = pca_swapped.explained_variance_ratio_[0] * 100
var2_pct_swapped = pca_swapped.explained_variance_ratio_[1] * 100

# (D) 計算 Loadings 箭頭的縮放因子
max_score_swapped = np.max([np.abs(pc1_scores_swapped).max(), np.abs(pc2_scores_swapped).max()])
max_loading_swapped = np.max([np.abs(pc1_loadings_swapped).max(), np.abs(pc2_loadings_swapped).max()])
scale_factor_swapped = (max_score_swapped / max_loading_swapped) * 0.9

# 4. 開始繪圖
# --------------------------------------------------
# 確保中文字型已設定
if 'font_name' in globals():
    plt.rcParams['font.family'] = font_name

fig, ax = plt.subplots(figsize=(14 * 0.8, 11 * 0.8))

# ➗️ (A) 繪製 Score Plot (年份投影點)
ax.scatter(
    pc1_scores_swapped, pc2_scores_swapped,
    c='#e06666', alpha=0.6, s=70, edgecolors='white',
    label='年份投影點'
)

# ✍️ 標註年份 (數字)
for i, txt in enumerate(year_names):
    ax.annotate(
        txt, (pc1_scores_swapped[i], pc2_scores_swapped[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#b30000', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的縣市箭頭向量)
arrow_color_swapped = '#2b5c8f' # 與點形成對比的顏色
for i in range(len(county_names_swapped)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings_swapped[i] * scale_factor_swapped
    end_y = pc2_loadings_swapped[i] * scale_factor_swapped

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color_swapped, alpha=0.5, linewidth=1,
        head_width=max_score_swapped * 0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (縣市) - 根據箭頭方向微調文字對齊方式
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(county_names_swapped[i]),
        color=arrow_color_swapped, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

ax.set_title(
    f'主成分分析雙標圖',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1 ', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

plt.savefig('pca_corr_biplot_years_by_counties.png', dpi=300, bbox_inches='tight')
plt.show()

### PCA using cov

In [ ]:
# ==========================================
# 1. 執行 PCA (qithout 標準化)
# ==========================================
pca_cov = PCA()
pca_cov.fit(pivot_table_cleaned)

# ✨ 使用 f"{x:.2%}" 自動將小數轉換為百分比並四捨五入至小數點後兩位
explained_variance_pct = [f"{x:.2%}" for x in pca_cov.explained_variance_ratio_[:7]]
cumulative_variance_pct = [f"{x:.2%}" for x in np.cumsum(pca_cov.explained_variance_ratio_)[:7]]

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前7個):", explained_variance_pct)
print("累積解釋變異比 (前7個):", cumulative_variance_pct)
print("-" * 38)

#### Scree Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 取得特徵值 (λ)
# 這裡使用共變異數矩陣的 PCA 結果，若要用相關係數矩陣請改用 pca_corr
eigenvalues = pca_cov.explained_variance_

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(eigenvalues) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製陡坡圖的折線與點
plt.plot(
    pc_num,
    eigenvalues,
    marker='o',
    linestyle='-',
    color='#2b5c8f',
    linewidth=2,
    markersize=6,
    label='Eigenvalue'
)

plt.title('PCA Scree Plot - Covariance Matrix', fontsize=16)
plt.xlabel('Principal Component', fontsize=13)
plt.ylabel('Eigenvalue (λ)', fontsize=13)

# 讓 X 軸的刻度整齊顯示每一個 PC 編號
plt.xticks(pc_num)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Variance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算累積解釋變異量比率 (乘以 100 轉為百分比)
cum_variance_ratio = np.cumsum(pca_cov.explained_variance_ratio_) * 100

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_variance_ratio) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製累積變異量的折線與點
plt.plot(
    pc_num,
    cum_variance_ratio,
    marker='s',          # 使用方形標記區隔之前的陡坡圖
    linestyle='-',
    color='#e06666',     # 使用顯眼的暖色系
    linewidth=2,
    markersize=6,
    label='Cumulative Explained Variance'
)

# ✨ 加上 80% 的基準輔助線 (通常實務上會看 80% 或 90% 的斷點)
plt.axhline(y=90, color='gray', linestyle='--', alpha=0.7, label='90% Threshold')

# 標註前幾個 PC 的具體數值（讓圖表更直覺）
# 這邊標註前 3 個成分作為範例
for i in range(3):
    plt.annotate(
        f"{cum_variance_ratio[i]:.1f}%",
        (pc_num[i], cum_variance_ratio[i]),
        textcoords="offset points",
        xytext=(0,10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# 圖表細節設定
plt.title('PCA (Covariance) Cumulative Explained Variance', fontsize=16)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

# 讓 X、Y 軸的刻度更貼切
plt.xticks(pc_num)
plt.ylim(0, 105) # Y 軸範圍設在 0~105% 比較有留白美感

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_cumulative_variance_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Var: Cov vs Corr

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算兩者的累積解釋變異量比率 (%)
cum_cov = np.cumsum(pca_cov.explained_variance_ratio_) * 100
cum_corr = np.cumsum(pca_corr.explained_variance_ratio_) * 100

# 設定 X 軸編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_cov) + 1)

# 2. 開始畫圖
plt.figure(figsize=(12, 7))

# 📈 繪製 共變異數矩陣 (Covariance) - 藍色線
plt.plot(
    pc_num, cum_cov,
    marker='o', linestyle='-', color='#2b5c8f', linewidth=2.5, markersize=7,
    label='Covariance (Not standardized)'
)

# 📈 繪製 相關係數矩陣 (Correlation) - 紅色線
plt.plot(
    pc_num, cum_corr,
    marker='s', linestyle='--', color='#e06666', linewidth=2.5, markersize=7,
    label='Correlation (Standardized)'
)

# 🎯 3. 精準標註前三個點的數值

# 共變異數矩陣 (pca_cov)：數字放【上面】 (xytext 設為正值)
for i in range(3):
    plt.annotate(
        f"{cum_cov[i]:.2f}%",
        (pc_num[i], cum_cov[i]),
        textcoords="offset points",
        xytext=(0, 10),           # 往上推 10 點
        ha='center', va='bottom',
        color='#2b5c8f', fontweight='bold', fontsize=10
    )

# 相關係數矩陣 (pca_corr)：數字放【下面】 (xytext 設為負值)
for i in range(3):
    plt.annotate(
        f"{cum_corr[i]:.2f}%",
        (pc_num[i], cum_corr[i]),
        textcoords="offset points",
        xytext=(0, -18),          # 往下推 18 點，避開點與線
        ha='center', va='top',
        color='#e06666', fontweight='bold', fontsize=10
    )

# ✨ 加上常用的 80% 與 90% 門檻參考線
plt.axhline(y=90, color='gray', linestyle=':', alpha=0.6, label='90% Threshold')
plt.axhline(y=80, color='gray', linestyle='-.', alpha=0.4, label='80% Threshold')

# 圖表美化與細節設定
plt.title('PCA Cumulative Explained Variance : Covariance vs Correlation', fontsize=16, pad=15)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

plt.xticks(pc_num)
plt.ylim(min(min(cum_cov), min(cum_corr)) - 5, 105) # 動態留白，畫面更好看

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_comparison_cumulative_variance.png', dpi=300, bbox_inches='tight')
plt.show()

## K-Means

#### 手肘法 (Elbow Method)

這個方法是去計算每個點到其所屬群集中心的距離平方和（SSE，在 sklearn 中稱為 inertia_）。

當 $k$ 越大，SSE 一定會越小。我們要找的是「下降斜率突然變平緩的那個轉折點」（就像手肘一樣）。

In [ ]:
sse = []
k_range = range(1, 11)
for i in k_range:
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    km.fit(scaled_data)
    sse.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker='o', color='#2b5c8f', linewidth=2)
plt.title('Elbow Method For Optimal k', fontsize=14)
plt.xlabel('Number of clusters (k)')
plt.ylabel('SSE (Inertia)')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('kmeans_elbow_plot.png', dpi=300)
plt.show()

#### 輪廓分數（Silhouette Score）

衡量的是「群體內的凝聚度」與「群體間的分離度」。

分數範圍在 $-1$ 到 $+1$ 之間：

* 分數越接近 1：代表分群非常完美，同群的點很近，異群的點很遠。
* 分數接近 0：代表群體邊界重疊，分得不夠好。

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range = range(2, 11) # 輪廓分數必須從 k=2 開始計算
for i in k_range:
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    labels = km.fit_predict(scaled_data)
    silhouette_scores.append(silhouette_score(scaled_data, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, marker='s', color='#e06666', linewidth=2)
plt.title('Silhouette Score For Optimal k', fontsize=14)
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('kmeans_silhouette_plot.png', dpi=300)
plt.show()

#### 比較 $$k=2,3,4,5$$

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ==========================================
# 🎯 在這裡自由修改你的 k 值 (例如：2, 3, 4, 5)
k = 4
# ==========================================

# 1. 執行 K-Means 分群
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

# 2. ✨ 動態生成 Group 名稱清單與對照表 (根據 k 自動產生 A, B, C, D, E...)
# chr(65) 是 'A', chr(66) 是 'B'，以此類推
groups_list = [f'Group {chr(65 + i)}' for i in range(k)]
cluster_map = {i: groups_list[i] for i in range(k)}
encoded_labels = [cluster_map[label] for label in kmeans_labels]

# 3. 建立畫圖用的 DataFrame 與 PCA 降維
counties_names = pivot_table_cleaned.index
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_data)

df_kmeans_plot = pd.DataFrame({
    'PC1': components_kmeans[:, 0],
    'PC2': -components_kmeans[:, 1],  # Y 軸翻轉對齊
    'County': counties_names,
    'Cluster': encoded_labels
})

# 4. 繪圖 (顏色調色盤改用 'Set1'，在 k 較少或到 5 時，視覺對比度都極佳)
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_kmeans_plot,
    x='PC1',
    y='PC2',
    hue='Cluster',
    hue_order=groups_list,  # ✨ 自動帶入動態群組順序 ['Group A', 'Group B', ...]
    palette='Set1',        # ✨ 改用 'Set1' 高對比調色盤，分 2~5 群都非常漂亮
    s=200,
    alpha=0.9,
    edgecolor='black'
)

# 標註縣市名稱 (中文字)
for i in range(len(df_kmeans_plot)):
    plt.annotate(
        df_kmeans_plot['County'].iloc[i],
        (df_kmeans_plot['PC1'].iloc[i], df_kmeans_plot['PC2'].iloc[i]),
        textcoords='offset points',
        xytext=(0, 12),
        ha='center',
        fontsize=11,
        fontname=prop.get_name() if 'prop' in locals() else None,
        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.3)
    )

# 設定標題和座標軸 (百分比自動抓取現場 PCA 結果)
plt.title(f'K-Means Clustering (k={k})', fontsize=16)
plt.xlabel(f'PC1 ({pca_kmeans.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca_kmeans.explained_variance_ratio_[1]*100:.2f}%)')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(title='Clusters', loc='best')

# 💾 儲存圖片 (檔名也會自動帶入 k 值)
plt.savefig(f'kmeans_pca_clusters_k{k}.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. ✨ 動態輸出各群組的縣市名單
print(f"\n--- Cluster Assignments (k={k}) ---")
for group in groups_list:
    members = df_kmeans_plot[df_kmeans_plot['Cluster'] == group]['County'].tolist()
    print(f"{group}: {'、'.join(members)}")



```
--- Cluster Assignments (k=2) ---
Group A: 台中市、台北市、台南市、台東縣、嘉義市、嘉義縣、基隆市、宜蘭縣、彰化縣、新北市、新竹市、桃園市、澎湖縣、花蓮縣、雲林縣、高雄市
Group B: 南投縣、屏東縣、新竹縣、苗栗縣、金門縣
--- Cluster Assignments (k=3) ---
Group A: 台北市、基隆市、新北市
Group B: 台中市、台南市、台東縣、嘉義市、嘉義縣、宜蘭縣、彰化縣、新竹市、桃園市、澎湖縣、花蓮縣、雲林縣、高雄市
Group C: 南投縣、屏東縣、新竹縣、苗栗縣、金門縣
--- Cluster Assignments (k=4) ---
Group A: 台中市、台南市、台東縣、嘉義市、嘉義縣、彰化縣、新竹市、桃園市、澎湖縣、花蓮縣、雲林縣、高雄市
Group B: 金門縣
Group C: 台北市、基隆市、宜蘭縣、新北市
Group D: 南投縣、屏東縣、新竹縣、苗栗縣
--- Cluster Assignments (k=5) ---
Group A: 台中市、台南市、台東縣、嘉義市、嘉義縣、宜蘭縣、彰化縣、新竹市、桃園市、花蓮縣、雲林縣、高雄市
Group B: 金門縣
Group C: 南投縣、屏東縣、新竹縣、苗栗縣
Group D: 台北市、基隆市、新北市
Group E: 澎湖縣
```



## Hierarchical Clustering

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.font_manager as fm
import os

# Standardize data - Handling NaNs (e.g., Lienchiang County)
clustering_data_clean = pivot_table_age_std.dropna()

scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(clustering_data_clean)

# Calculate Pearson correlation-based distance
pearson_distances = pdist(scaled_clustering_data, metric='correlation')
if not np.all(np.isfinite(pearson_distances)):
    pearson_distances = np.nan_to_num(pearson_distances, nan=1.0)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 初始化參數
methods = ['single', 'complete', 'average', 'ward']
k_range = range(2, 11)
results = []

# 2. 巢狀迴圈計算分數
for k in k_range:
    for method in methods:
        clusterer = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = clusterer.fit_predict(scaled_clustering_data)
        score = silhouette_score(scaled_clustering_data, labels)
        results.append({
            'k': k,
            'Method': method.capitalize(),
            'Silhouette Score': score
        })

# 3. 轉換為 DataFrame
df_comparison = pd.DataFrame(results)

# 4. 建立樞紐表以便觀察
df_pivot = df_comparison.pivot(index='k', columns='Method', values='Silhouette Score')
print("--- Silhouette Score Comparison (k=2 to 10) ---")
display(df_pivot)

# 5. 繪製趨勢圖
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_comparison, x='k', y='Silhouette Score', hue='Method', marker='o')
plt.title('Silhouette Score Trends by Linkage Methods', fontsize=14)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(list(k_range))
plt.legend(title='Linkage Method')

# 💾 ✨ 新增：儲存圖片（必須在 plt.show() 之前）
# dpi=300 確保圖片印刷或放入簡報時非常清晰
# bbox_inches='tight' 可以防止標籤或圖例被切到
plt.savefig('HClust_silhouette_score_trends.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering

# 1. 準備必要數據
counties_names = pivot_table_cleaned.index
methods = ['single', 'complete', 'average', 'ward']
k_targets = [2, 3, 4, 5]

# 2. 建立基準 DataFrame (以縣市名稱為 Index)
df_cluster_results = pd.DataFrame(index=counties_names)

# 3. 巢狀迴圈跑完 16 種組合，並直接塞入 DataFrame
for method in methods:
    for k in k_targets:
        # 初始化層次聚類模型
        clusterer = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = clusterer.fit_predict(scaled_clustering_data)

        # 動態生成 Group A, B, C... 的名稱對應
        groups_list = [f'Group {chr(65 + i)}' for i in range(k)]
        encoded_labels = [groups_list[label] for label in labels]

        # 欄位命名規則，例如: Ward_k5, Average_k2
        col_name = f"{method.capitalize()}_k{k}"

        # 寫入大表格中
        df_cluster_results[col_name] = encoded_labels

# 4. 排序欄位，讓表格由 k=2 到 k=5 依序排列，比較好閱讀
ordered_columns = [f"{m.capitalize()}_k{k}" for k in k_targets for m in methods]
df_cluster_results = df_cluster_results[ordered_columns]

# 5. 印出完整大表格
print("=== 4種 Linkage Methods 於 k=2,3,4,5 之實體分群結果大表 ===")
# 使用 pd.option_context 可以確保在 Jupyter 中完整秀出所有欄位不被隱藏
with pd.option_context('display.max_columns', None, 'display.expand_frame_repr', False):
    display(df_cluster_results)

# 6. 自動將這份珍貴的對照表存成 CSV，方便你直接拉進 Excel 做報告
df_cluster_results.to_csv('hierarchical_clustering_comprehensive_results.csv', encoding='utf-8-sig')
print("\n💾 完整對照表已成功儲存為 'hierarchical_clustering_comprehensive_results.csv' !")

#### Dendrogram

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.cluster.hierarchy import linkage, dendrogram

# 1. 準備必要數據 (請確保此時使用的數據為歐氏距離專用的標準化資料)
county_names = pivot_table_cleaned.index.tolist()
linkage_methods = ['single', 'complete', 'average', 'ward']

# 中文字型安全設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 2. 開始依序繪製四張獨立的樹狀圖
for method in linkage_methods:
    plt.figure(figsize=(13, 7))

    # 現場計算歐氏距離的階層結合矩陣 Z
    Z = linkage(scaled_clustering_data, method=method, metric='euclidean')

    # 3. 🎯 依據你的統計表格，動態計算對應的切線高度與染色門檻
    # Z[-1, 2] 是 2->1 群的高度, Z[-2, 2] 是 3->2 群的高度，以此類推
    h_k2 = (Z[-1, 2] + Z[-2, 2]) / 2  # k=2 的中點切線
    h_k4 = (Z[-3, 2] + Z[-4, 2]) / 2  # k=4 的中點切線
    h_k5 = (Z[-4, 2] + Z[-5, 2]) / 2  # k=5 的中點切線

    # 設定指定的染色門檻 (color_threshold)
    if method == 'ward':
        current_threshold = h_k5  # Ward 法染色切在 5 群
        second_k_label = f'2nd Best: k=5 (Height: {h_k5:.2f})'
        second_k_height = h_k5
    else:
        current_threshold = h_k4  # Single, Complete, Average 染色切在 4 群
        second_k_label = f'2nd Best: k=4 (Height: {h_k4:.2f})'
        second_k_height = h_k4

    # 4. 繪製樹狀圖本體
    dendrogram(
        Z,
        labels=county_names,
        color_threshold=current_threshold,  # 套用動態指定的群數染色
        orientation='top',
        leaf_rotation=45,
        leaf_font_size=11,
        distance_sort='descending'
    )

    # 5. 🗺️ 繪製兩條決策輔助切線
    # 🔴 畫出最高分 (k=2) 的紅色實線
    plt.axhline(y=h_k2, color='#d9383a', linestyle='-', linewidth=2.0,
                label=f'Best: k=2 (Height: {h_k2:.2f})')

    # 🔵 畫出第二高分 (k=4 或 k=5) 的藍色虛線
    plt.axhline(y=second_k_height, color='#2a7ae2', linestyle='--', linewidth=1.8,
                label=second_k_label)

    # 6. 圖表細節美化
    plt.title(f'Hierarchical Clustering Dendrogram (Euclidean - {method.capitalize()} Linkage)',
              fontsize=15, pad=15, fontweight='bold')
    plt.xlabel('Counties / Cities', fontsize=12)
    plt.ylabel('Distance Threshold', fontsize=12)
    plt.grid(axis='y', linestyle=':', alpha=0.5)
    plt.legend(title='Silhouette Decisions', loc='upper left', fontsize=10)
    plt.tight_layout()

    # 7. 💾 自動儲存高解析度圖片 (檔名包含對應的 linkage method)
    plt.savefig(f'dendrogram_euclidean_{method}_two_cutoffs.png', dpi=300, bbox_inches='tight')
    plt.show()

## Map visualization

In [ ]:
import geopandas as gpd
import os
from osgeo import gdal

# Set GDAL config option to restore or create missing .shx file
# Moved outside the try block to ensure it always takes effect early.
gdal.SetConfigOption('SHAPE_RESTORE_SHX', 'YES')

# 1. Define potential paths for the shapefile
shp_paths = '/content/drive/MyDrive/多變量分析/直轄市、縣(市)界線1140318/COUNTY_MOI_1140318.shp'

try:
    # Load the shapefile and project to WGS84
    taiwan_map = gpd.read_file(found_shp_file_path).to_crs(epsg=4326)
    # Dynamically find the county name column
    county_col = 'COUNTYNAME' if 'COUNTYNAME' in taiwan_map.columns else \
                  [c for c in taiwan_map.columns if 'NAME' in c.upper()][0]
    print(f"Shapefile loaded successfully from: {found_shp_file_path}")
    print(f"Using county column: {county_col}")
    # display(taiwan_map.head()) # Uncomment for inspection if needed
except Exception as e:
    print(f"An error occurred while reading the shapefile: {e}")
    taiwan_map = None # Ensure taiwan_map is None on error
    county_col = None

#### K-Means

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from sklearn.cluster import KMeans

# 排除連江縣
taiwan_map_clean = taiwan_map[taiwan_map[county_col] != '連江縣'].copy()
taiwan_map_clean[county_col] = taiwan_map_clean[county_col].str.replace('臺', '台', 'one' if hasattr(str, 'replace_old') else -1)

# 確保沿用你先前的字型設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 確保縣市名稱清單 (對齊 K-means 順序)
counties_names = pivot_table_cleaned.index

# 固定群組顏色
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# ==========================================
# 2. 進入迴圈，分別繪製 k=2 與 k=5
# ==========================================
for k_val in [2, 5]:

    # 執行 K-Means 分群
    kmeans = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)

    # 動態生成群組標籤
    groups_list = [f'Group {chr(65 + i)}' for i in range(k_val)]
    cluster_map = {i: groups_list[i] for i in range(k_val)}
    encoded_labels = [cluster_map[l] for l in labels]

    # 建立分群對照 DataFrame
    df_clusters = pd.DataFrame({
        'County': counties_names,
        'Cluster': encoded_labels
    })

    # 合併地圖與分群數據
    map_data = taiwan_map_clean.merge(df_clusters, left_on=county_col, right_on='County', how='left')
    map_data['color'] = map_data['Cluster'].map(color_dict).fillna('#cbd5e1')

    # ==========================================
    # 3. 開始建構圖表與子母畫面
    # ==========================================
    fig, ax_main = plt.subplots(figsize=(10, 12))

    # 🗺️ (A) 繪製主圖
    map_data.plot(ax=ax_main, color=map_data['color'], edgecolor='black', linewidth=0.4)

    # ✨ 關鍵修改：將經度左界從 120.0 放寬到 119.0，為左側小地圖拉出完美腹地
    ax_main.set_xlim(119.0, 122.3)
    ax_main.set_ylim(21.8, 25.4)

    ax_main.set_title(f'Taiwan K-Means Map\n(k={k_val})', fontsize=16, pad=20, fontweight='bold')
    ax_main.set_xlabel('Longitude (經度)', fontsize=11)
    ax_main.set_ylabel('Latitude (緯度)', fontsize=11)
    ax_main.grid(True, linestyle='--', alpha=0.3)

    for idx, row in map_data[map_data[county_col] != '金門縣'].iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax_main.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=12,
                             bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

    # 🗺️ (B) 嵌入金門小地圖
    # ✨ 改用 ax_main.inset_axes，[左邊距, 下邊距, 寬度, 高度] 比例完全鎖在主圖內
    ax_kinmen = ax_main.inset_axes([0.05, 0.72, 0.22, 0.22])
    kinmen_data = map_data[map_data[county_col] == '金門縣']
    kinmen_data.plot(ax=ax_kinmen, color=kinmen_data['color'], edgecolor='black', linewidth=0.5)
    ax_kinmen.set_xlim(118.1, 118.6)
    ax_kinmen.set_ylim(24.2, 24.6)
    ax_kinmen.set_title('金門縣', fontsize=12, fontweight='bold', color='#1e293b')
    ax_kinmen.set_xticks([])
    ax_kinmen.set_yticks([])

    # ==========================================
    # 4. 手動客製化圖例 (Legend)
    # ==========================================
    legend_elements = [
        Patch(facecolor=color_dict[group], edgecolor='black', linewidth=0.5, label=group)
        for group in groups_list
    ]
    ax_main.legend(handles=legend_elements, title='Clusters', loc='lower right', fontsize=12, title_fontsize=13)

    # 💾 儲存高解析度地圖
    plt.savefig(f'map_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
    plt.show()

print("\n🎉 地圖（k=2, k=5）已繪製完成")

#### H-Clustering

In [ ]:
# 讀取上一階段打包好的 16 種組合分群結果 (以縣市名稱為 Index)
df_hierarchical = pd.read_csv('hierarchical_clustering_comprehensive_results.csv', index_col=0)

# 確保沿用你的中文字型設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 固定群組高對比顏色面版
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# ==========================================
# 2. 定義方案 A 與方案 B 的地圖任務清單
# 欄位命名規則必須完美對齊 CSV 欄位（例如：'Single_k2', 'Ward_k5'）
# ==========================================
map_tasks = [
    # 方案 A：所有方法皆看 k = 2
    {'scenario': 'A', 'method': 'Single',   'k': 2},
    {'scenario': 'A', 'method': 'Complete', 'k': 2},
    {'scenario': 'A', 'method': 'Average',  'k': 2},
    {'scenario': 'A', 'method': 'Ward',     'k': 2},

    # 方案 B：指定最佳/次佳群數 (前三者 k=4, Ward k=5)
    {'scenario': 'B', 'method': 'Single',   'k': 4},
    {'scenario': 'B', 'method': 'Complete', 'k': 4},
    {'scenario': 'B', 'method': 'Average',  'k': 4},
    {'scenario': 'B', 'method': 'Ward',     'k': 5},
]

# ==========================================
# 3. 進入自動化自動繪圖迴圈 (共 8 張圖)
# ==========================================
for task in map_tasks:
    scenario = task['scenario']
    method = task['method']
    k_val = task['k']

    # 對齊 CSV 的欄位名稱
    csv_col_name = f"{method}_k{k_val}"

    # 建立暫時的對照 DataFrame 準備進行 Merge
    df_temp = pd.DataFrame({
        'County': df_hierarchical.index,
        'Cluster': df_hierarchical[csv_col_name]
    })

    # 地圖與數據結合
    map_data = taiwan_map_clean.merge(df_temp, left_on=county_col, right_on='County', how='left')
    map_data['color'] = map_data['Cluster'].map(color_dict).fillna('#cbd5e1')

    # 建立畫布
    fig, ax_main = plt.subplots(figsize=(10, 12))

    # 🗺️ (A) 繪製主圖 (範圍包含台灣本島與澎湖縣)
    map_data.plot(ax=ax_main, color=map_data['color'], edgecolor='black', linewidth=0.3)
    ax_main.set_xlim(119.0, 122.3)  # 完美容納澎湖的寬闊經度
    ax_main.set_ylim(21.8, 25.4)

    # 動態精美標題
    ax_main.set_title(f'Taiwan Hierarchical Clustering Map\n{method} Linkage (k={k_val})',
                      fontsize=15, pad=15, fontweight='bold')
    ax_main.set_xlabel('Longitude (經度)', fontsize=11)
    ax_main.set_ylabel('Latitude (緯度)', fontsize=11)
    ax_main.grid(True, linestyle='--', alpha=0.3)

    for idx, row in map_data[map_data[county_col] != '金門縣'].iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax_main.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=12,
                             bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

    # 🗺️ (B) 僅保留金門小地圖 (利用 inset_axes 舒適安放於左上角海域)
    ax_kinmen = ax_main.inset_axes([0.05, 0.72, 0.22, 0.22])
    kinmen_data = map_data[map_data[county_col] == '金門縣']
    kinmen_data.plot(ax=ax_kinmen, color=kinmen_data['color'], edgecolor='black', linewidth=0.4)
    ax_kinmen.set_xlim(118.1, 118.6)
    ax_kinmen.set_ylim(24.2, 24.6)
    ax_kinmen.set_title('金門縣', fontsize=12, fontweight='bold', color='#1e293b')
    ax_kinmen.set_xticks([])
    ax_kinmen.set_yticks([])

    # ==========================================
    # 4. 手動客製化圖例 (僅顯示該 k 值範圍內的 Group)
    # ==========================================
    groups_list = [f'Group {chr(65 + i)}' for i in range(k_val)]
    legend_elements = [
        Patch(facecolor=color_dict[group], edgecolor='black', linewidth=0.5, label=group)
        for group in groups_list
    ]
    ax_main.legend(handles=legend_elements, title=f'Clusters (k={k_val})', loc='lower right', fontsize=12, title_fontsize=13)

    # 💾 5. 自動儲存高解析度圖片，檔名格式：map_hierarchical_ScenarioA_Single_k2.png
    file_name = f'map_hierarchical_{method}_k{k_val}.png'
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"📸 已成功儲存地圖：{file_name}")

print("\n✨ 地圖已全部繪製並儲存完畢。")